# Análisis Exploratorio de Datos (EDA) — Afiliados al SNP

**Curso:** Big Data — UNAJ  
**Docente:** Leibnihtz Abrahan Ayamamani Choque  
**Semestre:** 2026-2  
**Dataset:** Base de Datos de Afiliados al Sistema Nacional de Pensiones (SNP) — Diciembre 2025  
**Fuente:** Superintendencia de Banca, Seguros y AFP (SBS)  
**Tamaño:** ~424 MB (CSV)

---

## Objetivos del Notebook

1. **Cargar y perfilar** un dataset real de escala Big Data (~424 MB)
2. **Decodificar** variables categóricas codificadas numéricamente
3. **Explorar** distribuciones, correlaciones y patrones
4. **Exportar** visualizaciones para la presentación de clase

---

## 0. Setup e Importaciones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
import time

warnings.filterwarnings('ignore')

# Estilo global
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.3
})

# Paleta del curso
NAVY      = '#1A3C5E'
STEEL     = '#2E86AB'
TEAL      = '#0E6B8B'
GOLD      = '#F4B942'
CORAL     = '#E55B4D'
GREEN     = '#27AE60'
LIGHT_BG  = '#F5F9FC'
PALETTE   = [STEEL, NAVY, GOLD, CORAL, GREEN, TEAL, '#8E44AD', '#E67E22']

# Directorio de salida para gráficos
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name: str, fig=None):
    """Guarda el gráfico actual en outputs/ como PNG de alta resolución."""
    path = OUTPUT_DIR / f'{name}.png'
    if fig:
        fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    else:
        plt.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'  ✅ Guardado: {path}')

print('✅ Librerías cargadas')
print(f'   Pandas {pd.__version__} | NumPy {np.__version__}')

## 1. Carga del Dataset

El archivo pesa ~424 MB. Usamos `dtype` optimizados y medimos el tiempo de carga.

In [ ]:
CSV_PATH = Path('../data/bronze/Afiliados_SNP_Diciembre2025.csv')

# Tipos optimizados para reducir uso de memoria
DTYPES = {
    'id_persona':       'int32',
    'sexoact':          'int8',
    'edadact':          'int8',
    'redadact':         'int8',
    'estcivil':         'int8',
    'dpto':             'int8',
    'aportante':        'int8',
    'tipo_afil':        'int8',
    'tipo_dep':         'int8',
    'tipo_trabajador':  'int8',
    'nro_aportes':      'int16',
}

print(f'⏳ Cargando {CSV_PATH.name} ({CSV_PATH.stat().st_size / 1e6:.0f} MB)...')
t0 = time.time()

df = pd.read_csv(
    CSV_PATH,
    dtype=DTYPES,
    low_memory=False
)

elapsed = time.time() - t0
print(f'✅ Cargado en {elapsed:.1f}s')
print(f'   Filas: {df.shape[0]:,}  |  Columnas: {df.shape[1]}')
print(f'   Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 2. Perfilado Inicial

In [ ]:
# Vista rápida
df.head(10)

In [ ]:
# Info de tipos y nulos
df.info(memory_usage='deep')

In [ ]:
# Estadísticas descriptivas
df.describe(include='all').T

In [ ]:
# Nulos por columna
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df) * 100).round(2)
resumen_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': pct_nulos})
resumen_nulos[resumen_nulos['Nulos'] > 0].sort_values('% Nulos', ascending=False)

In [ ]:
# Valores únicos por columna
pd.DataFrame({
    'Columna': df.columns,
    'Tipo': df.dtypes.values,
    'Únicos': [df[c].nunique() for c in df.columns],
    'Ejemplo_1': [df[c].dropna().iloc[0] if len(df[c].dropna()) > 0 else None for c in df.columns],
}).set_index('Columna')

## 3. Decodificación de Variables Categóricas

El dataset usa códigos numéricos. Creamos mapeos según el Diccionario de Datos.

In [ ]:
# === Mapeos del Diccionario de Datos ===

MAP_SEXO = {0: 'Mujer', 1: 'Hombre'}

MAP_REDAD = {
    1: 'Hasta 24',
    2: '25–34',
    3: '35–44',
    4: '45–54',
    5: '55–64',
    6: '65+'
}

MAP_ESTCIVIL = {
    1: 'Soltero/a',
    2: 'Casado/a',
    3: 'Viudo/a',
    4: 'Divorciado/a'
}

MAP_DPTO = {
    1: 'Amazonas', 2: 'Áncash', 3: 'Apurímac', 4: 'Arequipa',
    5: 'Ayacucho', 6: 'Cajamarca', 7: 'Callao', 8: 'Cusco',
    9: 'Huancavelica', 10: 'Huánuco', 11: 'Ica', 12: 'Junín',
    13: 'La Libertad', 14: 'Lambayeque', 15: 'Lima', 16: 'Loreto',
    17: 'Madre de Dios', 18: 'Moquegua', 19: 'Pasco', 20: 'Piura',
    21: 'Puno', 22: 'San Martín', 23: 'Tacna', 24: 'Tumbes',
    25: 'Ucayali', 26: 'Extranjero'
}

MAP_APORTANTE = {0: 'No', 1: 'Sí'}

MAP_TIPO_AFIL = {
    1: 'Obligatorio',
    2: 'Facultativo',
    3: 'Libre desafiliación'
}

MAP_TIPO_DEP = {
    1: 'Dependiente',
    2: 'Independiente'
}

# Aplicar mapeos (columnas _label para no perder el código original)
df['sexo']        = df['sexoact'].map(MAP_SEXO)
df['rango_edad']  = df['redadact'].map(MAP_REDAD)
df['est_civil']   = df['estcivil'].map(MAP_ESTCIVIL)
df['departamento']= df['dpto'].map(MAP_DPTO)
df['es_aportante']= df['aportante'].map(MAP_APORTANTE)
df['tipo_afiliacion'] = df['tipo_afil'].map(MAP_TIPO_AFIL)
df['tipo_dependencia']= df['tipo_dep'].map(MAP_TIPO_DEP)

print('✅ Variables decodificadas')
df[['sexo','rango_edad','est_civil','departamento','es_aportante',
    'tipo_afiliacion','tipo_dependencia']].head()

---
## 4. Visualizaciones para la Clase

Cada gráfico se exporta automáticamente a `../outputs/` para usar en la PPTX.

### 4.1 Distribución por Sexo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Conteos
sexo_counts = df['sexo'].value_counts()

# Pie chart
axes[0].pie(
    sexo_counts.values,
    labels=sexo_counts.index,
    autopct='%1.1f%%',
    colors=[STEEL, CORAL],
    startangle=90,
    textprops={'fontsize': 13, 'fontweight': 'bold'}
)
axes[0].set_title('Proporción por Sexo')

# Bar chart con valores
bars = axes[1].bar(sexo_counts.index, sexo_counts.values, color=[STEEL, CORAL], width=0.5)
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + h*0.01,
                 f'{h:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[1].set_title('Afiliados por Sexo')
axes[1].set_ylabel('Cantidad de afiliados')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Distribución de Afiliados SNP por Sexo — Dic 2025', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('01_distribucion_sexo', fig)
plt.show()

### 4.2 Distribución por Rango de Edad

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

orden_edad = ['Hasta 24', '25–34', '35–44', '45–54', '55–64', '65+']
edad_counts = df['rango_edad'].value_counts().reindex(orden_edad)

bars = ax.bar(orden_edad, edad_counts.values, color=PALETTE[:6], edgecolor='white', linewidth=0.5)

for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + h*0.01,
            f'{h:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_title('Afiliados SNP por Rango de Edad — Dic 2025', fontweight='bold', fontsize=14)
ax.set_xlabel('Rango de Edad')
ax.set_ylabel('Cantidad de afiliados')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
save_fig('02_distribucion_edad', fig)
plt.show()

### 4.3 Histograma de Edad Exacta

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(df['edadact'].dropna(), bins=80, color=STEEL, edgecolor='white', linewidth=0.3, alpha=0.9)
ax.axvline(df['edadact'].median(), color=CORAL, linewidth=2, linestyle='--',
           label=f'Mediana: {df["edadact"].median():.0f} años')
ax.axvline(df['edadact'].mean(), color=GOLD, linewidth=2, linestyle=':',
           label=f'Media: {df["edadact"].mean():.1f} años')

ax.set_title('Distribución de Edad de los Afiliados SNP — Dic 2025', fontweight='bold')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=11)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
save_fig('03_histograma_edad', fig)
plt.show()

### 4.4 Top 10 Departamentos

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

dpto_counts = df['departamento'].value_counts().head(10)
colors = [NAVY if i == 0 else STEEL for i in range(len(dpto_counts))]

bars = ax.barh(dpto_counts.index[::-1], dpto_counts.values[::-1], color=colors[::-1], height=0.6)

for bar in bars:
    w = bar.get_width()
    ax.text(w + w*0.01, bar.get_y() + bar.get_height()/2,
            f'{w:,.0f}', va='center', fontweight='bold', fontsize=10)

ax.set_title('Top 10 Departamentos — Afiliados SNP Dic 2025', fontweight='bold', fontsize=14)
ax.set_xlabel('Cantidad de afiliados')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
save_fig('04_top10_departamentos', fig)
plt.show()

### 4.5 Estado Civil

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ec_counts = df['est_civil'].value_counts()
bars = ax.bar(ec_counts.index, ec_counts.values, color=[STEEL, NAVY, GOLD, CORAL], width=0.5)

for bar in bars:
    h = bar.get_height()
    pct = h / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2, h + h*0.01,
            f'{h:,.0f}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_title('Afiliados SNP por Estado Civil — Dic 2025', fontweight='bold', fontsize=14)
ax.set_ylabel('Cantidad')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
save_fig('05_estado_civil', fig)
plt.show()

### 4.6 Aportantes vs No Aportantes (con cruce por Sexo)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Global
ap_counts = df['es_aportante'].value_counts()
axes[0].pie(
    ap_counts.values, labels=[f'{l}\n({v:,.0f})' for l, v in zip(ap_counts.index, ap_counts.values)],
    autopct='%1.1f%%', colors=[CORAL, GREEN],
    startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'}
)
axes[0].set_title('¿Aportó en el último mes?')

# Cruce sexo × aportante
cross = pd.crosstab(df['sexo'], df['es_aportante'], normalize='index') * 100
cross.plot(kind='bar', stacked=True, ax=axes[1], color=[CORAL, GREEN], edgecolor='white', width=0.5)
axes[1].set_title('Tasa de Aporte por Sexo')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].set_xlabel('')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='¿Aportante?', loc='upper right')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Indicador de Aporte — Afiliados SNP Dic 2025', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('06_aportantes_sexo', fig)
plt.show()

### 4.7 Tipo de Afiliación y Dependencia

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tipo afiliación
afil = df['tipo_afiliacion'].value_counts()
bars1 = axes[0].bar(afil.index, afil.values, color=[STEEL, GOLD, CORAL], width=0.5)
for bar in bars1:
    h = bar.get_height()
    axes[0].text(bar.get_x()+bar.get_width()/2, h+h*0.01,
                 f'{h:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[0].set_title('Tipo de Afiliación')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=15)
axes[0].spines[['top','right']].set_visible(False)

# Tipo dependencia
dep = df['tipo_dependencia'].value_counts()
bars2 = axes[1].bar(dep.index, dep.values, color=[NAVY, TEAL], width=0.4)
for bar in bars2:
    h = bar.get_height()
    axes[1].text(bar.get_x()+bar.get_width()/2, h+h*0.01,
                 f'{h:,.0f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
axes[1].set_title('Tipo de Dependencia')
axes[1].set_ylabel('Cantidad')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('Clasificación de Afiliados SNP — Dic 2025', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig('07_afiliacion_dependencia', fig)
plt.show()

### 4.8 Distribución de Remuneración (Último Aporte)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filtrar remuneraciones > 0 para el análisis
rem_positiva = df.loc[df['remuneracion'] > 0, 'remuneracion']

# Histograma (acotado al percentil 99 para legibilidad)
p99 = rem_positiva.quantile(0.99)
axes[0].hist(rem_positiva[rem_positiva <= p99], bins=80, color=STEEL, edgecolor='white', linewidth=0.3)
axes[0].axvline(rem_positiva.median(), color=CORAL, linewidth=2, linestyle='--',
                label=f'Mediana: S/ {rem_positiva.median():,.0f}')
axes[0].set_title('Distribución de Remuneración (hasta P99)')
axes[0].set_xlabel('Remuneración (S/)')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Boxplot por sexo
df_box = df.loc[(df['remuneracion'] > 0) & (df['remuneracion'] <= p99)]
df_box.boxplot(column='remuneracion', by='sexo', ax=axes[1],
               boxprops=dict(color=NAVY), medianprops=dict(color=CORAL, linewidth=2),
               flierprops=dict(marker='.', markersize=1, alpha=0.3))
axes[1].set_title('Remuneración por Sexo')
axes[1].set_xlabel('')
axes[1].set_ylabel('Remuneración (S/)')
fig.suptitle('Remuneración Declarada — Afiliados SNP Dic 2025', fontsize=14, fontweight='bold', y=1.02)

plt.tight_layout()
save_fig('08_remuneracion', fig)
plt.show()

# Estadísticas
print('\n📊 Estadísticas de Remuneración (S/) — solo aportes positivos:')
print(rem_positiva.describe().apply(lambda x: f'{x:,.2f}'))

### 4.9 Mapa de Calor — Edad × Departamento (Top 8)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

top8_dptos = df['departamento'].value_counts().head(8).index.tolist()
df_top = df[df['departamento'].isin(top8_dptos)]

heat_data = pd.crosstab(df_top['departamento'], df_top['rango_edad'])
heat_data = heat_data.reindex(columns=orden_edad)
heat_data = heat_data.loc[top8_dptos]  # mantener orden por volumen

sns.heatmap(
    heat_data, annot=True, fmt=',', cmap='YlGnBu',
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Cantidad de afiliados'}
)

ax.set_title('Concentración de Afiliados: Departamento × Rango de Edad (Top 8)', fontweight='bold', fontsize=13)
ax.set_xlabel('Rango de Edad')
ax.set_ylabel('Departamento')

plt.tight_layout()
save_fig('09_heatmap_dpto_edad', fig)
plt.show()

### 4.10 Número de Aportes Acumulados

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

nro_ap = df['nro_aportes'].dropna()
p95 = nro_ap.quantile(0.95)

ax.hist(nro_ap[nro_ap <= p95], bins=60, color=TEAL, edgecolor='white', linewidth=0.3, alpha=0.9)
ax.axvline(nro_ap.median(), color=GOLD, linewidth=2, linestyle='--',
           label=f'Mediana: {nro_ap.median():.0f} meses')
ax.axvline(nro_ap.mean(), color=CORAL, linewidth=2, linestyle=':',
           label=f'Media: {nro_ap.mean():.1f} meses')

ax.set_title('Distribución del Número de Meses Aportados (hasta P95) — SNP Dic 2025',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Meses de aportes')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=11)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
save_fig('10_nro_aportes', fig)
plt.show()

### 4.11 Remuneración Mediana por Departamento (Top 15)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

rem_dpto = (df.loc[df['remuneracion'] > 0]
            .groupby('departamento')['remuneracion']
            .median()
            .sort_values(ascending=True)
            .tail(15))

colors_rem = [NAVY if v == rem_dpto.max() else STEEL for v in rem_dpto.values]
bars = ax.barh(rem_dpto.index, rem_dpto.values, color=colors_rem, height=0.6)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 20, bar.get_y() + bar.get_height()/2,
            f'S/ {w:,.0f}', va='center', fontweight='bold', fontsize=10)

ax.set_title('Remuneración Mediana por Departamento (Top 15) — SNP Dic 2025',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Remuneración mediana (S/)')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
save_fig('11_remuneracion_dpto', fig)
plt.show()

### 4.12 Correlación entre Variables Numéricas

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

cols_num = ['edadact', 'aporte', 'remuneracion', 'nro_aportes', 'monto_aportes']
corr = df[cols_num].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=1, linecolor='white', ax=ax,
    xticklabels=['Edad', 'Aporte', 'Remuneración', 'Nro Aportes', 'Monto Aportes'],
    yticklabels=['Edad', 'Aporte', 'Remuneración', 'Nro Aportes', 'Monto Aportes']
)

ax.set_title('Matriz de Correlación — Variables Numéricas SNP', fontweight='bold', fontsize=13)

plt.tight_layout()
save_fig('12_correlacion', fig)
plt.show()

---
## 5. Resumen de Hallazgos Clave

Ejecutar esta celda al final para generar un resumen automático.

In [ ]:
print('=' * 70)
print('RESUMEN EDA — Afiliados SNP Diciembre 2025')
print('=' * 70)

n = len(df)
print(f'\n📌 Total de registros:       {n:>12,}')
print(f'📌 Columnas originales:      {df.shape[1] - 7:>12}')
print(f'📌 Memoria en RAM:           {df.memory_usage(deep=True).sum()/1e6:>12.1f} MB')

print(f'\n👫 Mujeres:                  {(df["sexoact"]==0).sum():>12,} ({(df["sexoact"]==0).mean()*100:.1f}%)')
print(f'👫 Hombres:                  {(df["sexoact"]==1).sum():>12,} ({(df["sexoact"]==1).mean()*100:.1f}%)')

print(f'\n🎂 Edad media:               {df["edadact"].mean():>12.1f} años')
print(f'🎂 Edad mediana:             {df["edadact"].median():>12.0f} años')

top_dpto = df['departamento'].value_counts().head(3)
print(f'\n🗺️  Top departamentos:')
for i, (dpto, cnt) in enumerate(top_dpto.items(), 1):
    print(f'   {i}. {dpto}: {cnt:,} ({cnt/n*100:.1f}%)')

pct_aportante = (df['aportante'] == 1).mean() * 100
print(f'\n💰 Aportantes último mes:    {pct_aportante:>12.1f}%')

rem_pos = df.loc[df['remuneracion'] > 0, 'remuneracion']
print(f'💰 Remuneración mediana:     S/ {rem_pos.median():>9,.0f}')
print(f'💰 Remuneración media:       S/ {rem_pos.mean():>9,.0f}')

print(f'\n📊 Meses aportados (media):  {df["nro_aportes"].mean():>12.1f}')
print(f'📊 Meses aportados (mediana):{df["nro_aportes"].median():>12.0f}')

print(f'\n✅ Gráficos exportados a: {OUTPUT_DIR.resolve()}')
print(f'   Archivos: {list(OUTPUT_DIR.glob("*.png"))}')
print('=' * 70)

---
## 6. Próximos Pasos

1. Ejecutar este notebook con el CSV completo
2. Revisar los gráficos en `../outputs/`
3. Generar la PPTX de clase con los gráficos exportados
4. Opcional: profundizar análisis por departamento (Puno, Lima) o brecha de género